In [1]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

# =====================================================
# Create output directories
# =====================================================

os.makedirs("images", exist_ok=True)
os.makedirs("curve_points", exist_ok=True)
os.makedirs("positive_points", exist_ok=True)
os.makedirs("negative_points", exist_ok=True)

# =====================================================
# Parameters
# =====================================================

N_SAMPLES = 5000

GRID_RES = 400
RANGE = 10.0

N_CURVE = 256
N_POS = 512
N_NEG = 512

EPS = 0.02

saved_count = 0

# =====================================================
# Generate dataset
# =====================================================

with open("labels.csv", mode="w", newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["filename", "a", "b", "c", "d", "e", "f"])

    while saved_count < N_SAMPLES:

        # ------------------------------------------
        # Random coefficients
        # ------------------------------------------

        a, b, c = np.random.uniform(-1, 1, 3)
        d, e = np.random.uniform(-2, 2, 2)
        f0 = np.random.uniform(-3, 3)

        if abs(a) + abs(b) + abs(c) < 0.25:
            continue

        # ------------------------------------------
        # Normalize coefficients
        # ------------------------------------------

        norm = np.sqrt(
            a*a + b*b + c*c +
            d*d + e*e + f0*f0
        )

        a /= norm
        b /= norm
        c /= norm
        d /= norm
        e /= norm
        f0 /= norm

        # ------------------------------------------
        # Evaluate on grid
        # ------------------------------------------

        x = np.linspace(-RANGE, RANGE, GRID_RES)
        y = np.linspace(-RANGE, RANGE, GRID_RES)

        X, Y = np.meshgrid(x, y)

        Z = (
            a * X**2 +
            b * X * Y +
            c * Y**2 +
            d * X +
            e * Y +
            f0
        )

        # Curve must intersect the grid

        if not (np.any(Z > 0) and np.any(Z < 0)):
            continue

        # ------------------------------------------
        # Extract contour
        # ------------------------------------------

        fig, ax = plt.subplots(figsize=(3, 3))

        cs = ax.contour(
            X,
            Y,
            Z,
            levels=[0],
            colors="black"
        )

        ax.axis("off")
        ax.set_aspect("equal")

        if len(cs.allsegs[0]) == 0:
            plt.close(fig)
            continue

        curve = np.vstack(cs.allsegs[0])

        if len(curve) < N_CURVE:
            plt.close(fig)
            continue

        # ------------------------------------------
        # Positive / Negative regions
        # ------------------------------------------

        pos_idx = np.argwhere(Z > EPS)
        neg_idx = np.argwhere(Z < -EPS)

        if len(pos_idx) < N_POS or len(neg_idx) < N_NEG:
            plt.close(fig)
            continue

        # ------------------------------------------
        # Sample curve points
        # ------------------------------------------

        curve_sel = np.random.choice(
            len(curve),
            N_CURVE,
            replace=False
        )

        curve_points = (
            curve[curve_sel] / RANGE
        ).astype(np.float32)

        # ------------------------------------------
        # Sample positive points
        # ------------------------------------------

        pos_sel = pos_idx[
            np.random.choice(
                len(pos_idx),
                N_POS,
                replace=False
            )
        ]

        positive_points = np.column_stack([
            X[pos_sel[:, 0], pos_sel[:, 1]],
            Y[pos_sel[:, 0], pos_sel[:, 1]]
        ])

        positive_points = (
            positive_points / RANGE
        ).astype(np.float32)

        # ------------------------------------------
        # Sample negative points
        # ------------------------------------------

        neg_sel = neg_idx[
            np.random.choice(
                len(neg_idx),
                N_NEG,
                replace=False
            )
        ]

        negative_points = np.column_stack([
            X[neg_sel[:, 0], neg_sel[:, 1]],
            Y[neg_sel[:, 0], neg_sel[:, 1]]
        ])

        negative_points = (
            negative_points / RANGE
        ).astype(np.float32)

        # ------------------------------------------
        # Save image
        # ------------------------------------------

        filename = f"sample_{saved_count:04d}.png"

        plt.savefig(
            os.path.join("images", filename),
            dpi=100,
            bbox_inches="tight",
            pad_inches=0
        )

        plt.close(fig)

        # ------------------------------------------
        # Save point sets
        # ------------------------------------------

        np.save(
            os.path.join(
                "curve_points",
                f"sample_{saved_count:04d}.npy"
            ),
            curve_points
        )

        np.save(
            os.path.join(
                "positive_points",
                f"sample_{saved_count:04d}.npy"
            ),
            positive_points
        )

        np.save(
            os.path.join(
                "negative_points",
                f"sample_{saved_count:04d}.npy"
            ),
            negative_points
        )

        # ------------------------------------------
        # Save coefficients
        # ------------------------------------------

        writer.writerow([
            filename,
            a,
            b,
            c,
            d,
            e,
            f0
        ])

        saved_count += 1

        if saved_count % 500 == 0:
            print(f"{saved_count}/{N_SAMPLES} samples generated.")

print("✅ Quadratic dataset generation completed.")

500/5000 samples generated.
1000/5000 samples generated.
1500/5000 samples generated.
2000/5000 samples generated.
2500/5000 samples generated.
3000/5000 samples generated.
3500/5000 samples generated.
4000/5000 samples generated.
4500/5000 samples generated.
5000/5000 samples generated.
✅ Quadratic dataset generation completed.
